# Kunskapskontroll 2 – Interaktiv datavisualisering med Plotly

**Josefina Nadafan**

I denna notebook undersöks hur Plotly kan användas för att skapa interaktiva visualiseringar av Gapminder-data. Pandas används för enkel filtrering och gruppering. Fokus ligger på hover-information, zoomning, filtrering och animation.

> Kör cellerna uppifrån och ned med **Shift + Enter**.

## Installation

Kör installationscellen en gång om biblioteken inte redan finns i din Jupyter-miljö. Efter installationen kan du behöva starta om kernel och sedan köra notebooken från början.

In [ ]:
%pip install pandas plotly matplotlib

## 1. Importera bibliotek

- **Pandas** används indirekt genom DataFrame-objektet och för filtrering/gruppering.
- **Plotly Express** används för de interaktiva diagrammen.
- **Matplotlib** används endast för en kort jämförelse med en statisk visualisering.

In [ ]:
import matplotlib.pyplot as plt
import plotly.express as px

# Länder från olika kontinenter för att göra jämförelsen tydlig
LANDER = ["Sweden", "Nigeria", "China", "United States", "Australia"]

## 2. Ladda och undersök datan

Gapminder-datan finns inbyggd i Plotly Express. Varje rad beskriver ett land under ett visst år och innehåller bland annat befolkning, förväntad livslängd och BNP per capita.

In [ ]:
df = px.data.gapminder()

print(f"Antal rader: {len(df)}")
print(f"Kolumner: {df.columns.tolist()}")
print(f"År: {df['year'].unique()}")
print(f"Kontinenter: {df['continent'].unique()}")

df.head()

## 3. Interaktivt linjediagram

Datan filtreras till fem länder. I Plotly kan användaren hovra för att se exakta värden, zooma och klicka på länder i legenden för att visa eller dölja linjer.

In [ ]:
df_lander = df[df["country"].isin(LANDER)]

fig_linje = px.line(
    df_lander,
    x="year",
    y="lifeExp",
    color="country",
    markers=True,
    title="Förväntad livslängd över tid för utvalda länder",
    labels={
        "year": "År",
        "lifeExp": "Förväntad livslängd (år)",
        "country": "Land",
    },
    hover_data={"country": True, "year": True, "lifeExp": ":.1f"},
)
fig_linje.show()

## 4. Interaktivt stapeldiagram

Först väljs det senaste året. Därefter används `groupby()` och `mean()` för att beräkna genomsnittlig livslängd per kontinent.

In [ ]:
senaste_aret = df["year"].max()
df_senaste = df[df["year"] == senaste_aret]
medel_per_kontinent = (
    df_senaste.groupby("continent", as_index=False)["lifeExp"].mean()
)

fig_stapel = px.bar(
    medel_per_kontinent,
    x="continent",
    y="lifeExp",
    color="continent",
    title=f"Genomsnittlig livslängd per kontinent ({senaste_aret})",
    labels={
        "continent": "Kontinent",
        "lifeExp": "Genomsnittlig livslängd (år)",
    },
    hover_data={"lifeExp": ":.1f"},
)
fig_stapel.update_layout(showlegend=False)
fig_stapel.show()

## 5. Interaktivt bubbeldiagram

Diagrammet visar flera variabler samtidigt: BNP per capita på x-axeln, livslängd på y-axeln, kontinent som färg och befolkning som bubbelstorlek. X-axeln är logaritmisk eftersom skillnaderna i BNP per capita är mycket stora.

In [ ]:
fig_bubbla = px.scatter(
    df_senaste,
    x="gdpPercap",
    y="lifeExp",
    color="continent",
    size="pop",
    hover_name="country",
    hover_data={
        "continent": True,
        "year": True,
        "gdpPercap": ":,.0f",
        "lifeExp": ":.1f",
        "pop": ":,",
    },
    log_x=True,
    size_max=60,
    title=f"BNP per capita och förväntad livslängd ({senaste_aret})",
    labels={
        "gdpPercap": "BNP per capita (logaritmisk skala)",
        "lifeExp": "Förväntad livslängd (år)",
        "continent": "Kontinent",
        "pop": "Befolkning",
        "year": "År",
    },
)
fig_bubbla.show()

## 6. Animation över tid

`animation_frame="year"` skapar ett steg för varje år. Med play-knappen kan användaren följa hur ländernas BNP, livslängd och befolkning förändras.

In [ ]:
fig_animation = px.scatter(
    df,
    x="gdpPercap",
    y="lifeExp",
    color="continent",
    size="pop",
    hover_name="country",
    animation_frame="year",
    animation_group="country",
    log_x=True,
    size_max=60,
    range_x=[100, 100000],
    range_y=[20, 90],
    title="BNP och förväntad livslängd över tid",
    labels={
        "gdpPercap": "BNP per capita (logaritmisk skala)",
        "lifeExp": "Förväntad livslängd (år)",
        "continent": "Kontinent",
        "pop": "Befolkning",
        "year": "År",
    },
)
fig_animation.show()

## 7. Filtrering med dropdown

Plotly skapar en dataserie (*trace*) per kontinent. Dropdown-menyn ändrar vilka serier som är synliga. Detta gör att användaren kan undersöka en kontinent i taget.

In [ ]:
fig_filter = px.scatter(
    df_senaste,
    x="gdpPercap",
    y="lifeExp",
    color="continent",
    size="pop",
    hover_name="country",
    log_x=True,
    size_max=60,
    title=f"Filtrera på kontinent ({senaste_aret})",
    labels={
        "gdpPercap": "BNP per capita (logaritmisk skala)",
        "lifeExp": "Förväntad livslängd (år)",
        "continent": "Kontinent",
        "pop": "Befolkning",
    },
)

knappar = [
    dict(
        label="Alla kontinenter",
        method="update",
        args=[{"visible": [True] * len(fig_filter.data)}],
    )
]

for vald_serie in fig_filter.data:
    synliga = [trace.name == vald_serie.name for trace in fig_filter.data]
    knappar.append(
        dict(
            label=vald_serie.name,
            method="update",
            args=[{"visible": synliga}],
        )
    )

fig_filter.update_layout(
    updatemenus=[
        dict(
            buttons=knappar,
            direction="down",
            showactive=True,
            x=1.0,
            y=1.15,
            xanchor="right",
        )
    ]
)
fig_filter.show()

## 8. Statisk och interaktiv visualisering

Samma data visas först med Matplotlib och sedan med Plotly. Matplotlib-versionen är en färdig bild. Plotly-versionen låter användaren zooma, hovra över punkter och visa eller dölja enskilda länder.

In [ ]:
# Statisk visualisering med Matplotlib
plt.figure(figsize=(9, 5))
for land in LANDER:
    delmangd = df_lander[df_lander["country"] == land]
    plt.plot(delmangd["year"], delmangd["lifeExp"], marker="o", label=land)

plt.title("Matplotlib: statisk visualisering")
plt.xlabel("År")
plt.ylabel("Förväntad livslängd (år)")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

In [ ]:
# Interaktiv visualisering av samma data med Plotly
fig_jamforelse = px.line(
    df_lander,
    x="year",
    y="lifeExp",
    color="country",
    markers=True,
    title="Plotly: samma data med interaktivitet",
    labels={
        "year": "År",
        "lifeExp": "Förväntad livslängd (år)",
        "country": "Land",
    },
    hover_data={"lifeExp": ":.1f"},
)
fig_jamforelse.show()

## Slutsats

Plotly gör det möjligt att presentera flera dimensioner av samma data och låta användaren utforska resultatet. Hover visar exakta värden, zoomning ger möjlighet att granska delar av ett diagram, dropdown-menyn filtrerar innehållet och animationen visar förändring över tid. En begränsning är att interaktiva diagram kräver en miljö som kan köra eller visa HTML och JavaScript, medan en statisk bild är enklare att använda i exempelvis utskrifter.

## Källor och dokumentation

- [Plotly Express – officiell dokumentation](https://plotly.com/python/plotly-express/)
- [Plotly – animationer](https://plotly.com/python/animations/)
- [Plotly – inbyggda dataset och Gapminder](https://plotly.com/python-api-reference/generated/plotly.data.html)
- [Pandas – GroupBy](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [Gapminder – data](https://www.gapminder.org/data/)

Källorna används för att förstå Plotlys diagram och animationer, Pandas gruppering samt innehållet i Gapminder-datan. En fullständig källförteckning kommer även att finnas i rapporten.